# Week 1 Day 2 Lab - Introducing UnderCover

## LLMs and Agents on a commercial task: insurance claims

A claims handler at UnderCover Travel Insurance has already decided each of the 25 claims in its queue. We send every claim to a few models, along with the customer's policy record and the policy wording. Each model replies with a decision of approve, decline or refer, a rationale and a payout. Then we compare the answers with the handlers' decisions and look at what each model cost.

At the end we give the same job to a LangChain agent that uses tools to look up the policy and record its decision.

## 4 parts:

PART 1: Examine the data

PART 2: Run with some models

PART 3: Optional: larger models and Jev

PART 4: An Agent via LangChain

In [41]:
!pip uninstall typing-extensions

Found existing installation: typing-extensions None


error: uninstall-no-record-file

× Cannot uninstall typing-extensions None
╰─> The package's contents are unknown: no RECORD file was found for typing-extensions.

hint: You might be able to recover from this via: pip install --force-reinstall --no-deps typing-extensions==4.15.0


In [36]:
!pip install typing-extensions==4.15.0

In [62]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
# from openrouter import OpenRouter
from pydantic import BaseModel, Field
import time

## Connect to OpenRouter - and please read note here

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

### NOTE:

If this is a new OpenRouter account, then you might get an error at some point from OpenRouter like: "new accounts are limited to 20 requests per minute for this model"

If you get that error, then add this line immediately after calling an LLM:

`time.sleep(3)`

and you should be in good shape.

In [2]:
os.getenv("CLOUD_ML_REGION")

'us-central1'

In [63]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

OpenRouter API key found


In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the cowboy?"}
]

response = openrouter.chat.completions.create(model="openai/gpt-6-luna", messages=messages)
display(Markdown(response.choices[0].message.content))

Arrr, a **cowboy** is literally a person who tends cattle, especially on horseback. As a symbol, the cowboy often represents **independence, self-reliance, adventure, and the American West**—though the image can also gloss over the history and hardships of life on the frontier.

If you mean “the cowboy” in a particular song, story, dream, or phrase, tell me which one and I’ll interpret it in context.

## Part 1: Load and examine the data

The seed data is in the repo's `data` folder. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id. The model is sent the claims, the policies and the policy wording.

By default we work with the queue of 25 claims. If you are using free models with a daily limit, set `TINY = True` to keep 5 of them.

In [5]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [6]:
TINY = True

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

5 claims, 5 labels, 50 policies
The policy wording is 1,369 words


In [7]:
claims[0]

{'claim_id': 'CLM-1007',
 'policy_id': 'POL-2007',
 'customer_id': 'CUS-3007',
 'claim_type': 'baggage',
 'submitted_on': '2026-07-22',
 'incident_date': '2026-07-13',
 'incident_location': 'Stockholm, Sweden',
 'summary': 'Phone dropped in the hotel shower',
 'story': 'I dropped my phone in the shower at my hotel in Stockholm on July 13. I know how that sounds. It was on the soap shelf playing music, I was singing along, and during the key change I knocked it off the shelf. It landed screen down in about an inch of water and stayed there longer than it should have, because I had shampoo in my eyes. A repair shop looked at it the next day and said the liquid damage was too extensive to repair. I am claiming the cost of the replacement phone, which was $1,099.',
 'items': [{'description': 'Smartphone, 256 GB, replacement cost',
   'category': 'electronics',
   'amount': 1099.0}],
 'amount_claimed': 1099.0,
 'documents': [{'name': 'Repair shop report',
   'type': 'repair_report',
   'con

Here is the handler's decision on that claim which we'll use to validate the LLM results.

In [8]:
labels[claims[0]["claim_id"]]

{'claim_id': 'CLM-1007',
 'decision': 'approve',
 'payout': 50.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-07-24',
 'reason': 'Accidental damage during the covered trip, with a repair report stating the phone cannot be repaired. The policy is Economy with no Electronics Plus, so the phone counts at the $150 per-item limit, and the $100 deductible comes off.',
 'reference': 'Policy wording, sections 5 and 10',
 'calculation': [{'step': 'Phone at the per-item limit', 'amount': 150.0},
  {'step': 'Less the deductible', 'amount': -100.0}],
 'tests': 'Payout arithmetic: a per-item limit far below the amount claimed',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'amusing',
 'tiny': True}

### Explore the data

It's always good to dig into the data before running experiments.

In [9]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1007,Phone dropped in the hotel shower,baggage,1099.0,routine,approve,50.0
1,CLM-1015,Flew home from Paris on arrival after leaving ...,trip_interruption,2340.0,routine,decline,0.0
2,CLM-1017,Backpack snatched by a scooter rider in Naples,baggage,830.0,routine,refer,0.0
3,CLM-1020,Prescription sunglasses stolen by a monkey at ...,baggage,240.0,manual,approve,240.0
4,CLM-1021,Tote bag snatched from her shoulder in Buenos ...,baggage,1103.0,hard,approve,825.0


Claim CLM-1015 might be a familiar example...

In [16]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Flew home from Paris on arrival after leaving a child at home**

We flew from Chicago to Paris on June 20 as a family of six. Somewhere over the Atlantic I realized that our youngest son, Kevin, who is eight, was not on the plane. We had overslept, we left for the airport in a rush and the head count in the van was wrong. I took the first available flight home the moment we landed in Paris. Kevin was fine. In his words, he had handled it. I am claiming my one-way fare home and my share of the prepaid hotel nights in Paris that I did not use.

**Handler:** decline $0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.


Here's another one that might ring a bell...

In [17]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Emergency dental treatment in Las Vegas for a missing front tooth**

I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger.

**Handler:** decline $0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.


This chart counts the claims of each type, split by the handler's decision.

In [18]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

Here's a breakdown by difficulty. The policy wording is enough to decide the routine and hard claims. The manual claims depend on the claims manual or the customer's history.

In [ ]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

## Part 2: running with some LLMs

Starting by defining a Pydantic class to describe the output so we can use Structured Outputs.

The rationale comes before the decision to ensure the model thinks through the decision before making it.

In [11]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt has the instructions and the whole policy wording. The user prompt has the customer's policy record and the claim, both as JSON.

In [12]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

### Calling the LLM

In [13]:
def decide(claim, model, effort):
    """Ask one model to decide one claim."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "effort": effort,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 1),
    }

In [ ]:
first = decide(claims[0], "openai/gpt-6-luna", "low")

first

# LLM call using anthropic/claude-haiku-5.5

In [65]:
model_haikku = "anthropic/claude-haiku-5.5"
model_sonnet = "anthropic/claude-sonnet-5.5"
model_opus = "anthropic/claude-opus-5.5"


In [64]:
## anthropic/claude-haiku-5.5

first = decide(claims[0], "anthropic/claude-haiku-5.5", "low")

first

{'model': 'anthropic/claude-haiku-5.5',
 'effort': 'low',
 'technique': 'LLM',
 'claim_id': 'CLM-1007',
 'decision': 'approve',
 'payout': 50.0,
 'rationale': 'Accidental damage to an electronic item during a covered trip falls under Baggage and Personal Effects (section 5), and the repair report and replacement receipt satisfy the accidental-damage documentation requirement. The Economy per-item limit is $150 and electronics are not raised without Electronics Plus, so the payout is capped at $150 with no deductible applied for this claim beyond the $100 Economy deductible, giving $50.',
 'cost': 0.0004874,
 'seconds': 3.3}

### Now running multiple models

In [16]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, effort, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model, effort)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model} at {effort}: {right} of {len(results)} right for ${cost:.4f} in {seconds:.0f} seconds")
    return results

In [17]:
luna_low = run("openai/gpt-6-luna", "low", claims, labels)

openai/gpt-6-luna at low: 4 of 5 right for $0.0008 in 13 seconds


In [18]:
luna_medium = run("openai/gpt-6-luna", "medium", claims, labels)

openai/gpt-6-luna at medium: 3 of 5 right for $0.0013 in 17 seconds


In [19]:
# Add your own experiments here!
luna_medium = run("openai/gpt-6-luna", "high", claims, labels)

openai/gpt-6-luna at high: 3 of 5 right for $0.0014 in 23 seconds


## Ansaribinhyder - Contribution 

In [67]:
#Haikku Low Medium and High

haikku_low = run(model_haikku, "low", claims, labels)
haikku_medium = run(model_haikku, "medium", claims, labels)
haikku_high = run(model_haikku, "high", claims, labels)

anthropic/claude-haiku-5.5 at low: 4 of 5 right for $0.0031 in 16 seconds
anthropic/claude-haiku-5.5 at medium: 3 of 5 right for $0.0037 in 23 seconds
anthropic/claude-haiku-5.5 at high: 4 of 5 right for $0.0043 in 29 seconds


In [20]:
gemini = run("google/gemini-3.5-flash-lite", "low", claims, labels)

google/gemini-3.5-flash-lite at low: 3 of 5 right for $0.0083 in 9 seconds


## Results by difficulty

This counts the claims each entry got right at each level of difficulty. Expect nearly all of the manual claims to be wrong for now.

In [68]:
scored = pd.DataFrame(haikku_low + haikku_medium + gemini)
scored.pivot_table(index=["model", "effort"], columns="difficulty", values="right", aggfunc="sum", sort=False)

difficulty                           routine  manual  hard
model                        effort                       
anthropic/claude-haiku-5.5   low           3       0     1
                             medium        3       0     0
google/gemini-3.5-flash-lite low           3       0     0

In [ ]:
scored = pd.DataFrame(luna_low + luna_medium + gemini)
scored.pivot_table(index=["model", "effort"], columns="difficulty", values="right", aggfunc="sum", sort=False)

difficulty                           routine  manual  hard
model                        effort                       
openai/gpt-6-luna            low           3       0     1
                             high          3       0     0
google/gemini-3.5-flash-lite low           3       0     0

## Where a model got it wrong

For each claim a model got wrong, this shows its rationale, the handler's reason and what the claim is there to test. Here we look at Luna at `low`. Pass in another set of results to see its misses.

In [ ]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(luna_low, labels)

**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The policy covers loss of personal belongings during the covered trip, subject to the per-item limit and the $0 First-plan deductible (sections 1, 5 and 10). However, the reported loss is theft, which section 5 requires to be reported to local police within 24 hours with a copy of the police report; the attached guide’s letter does not meet that requirement, so the claim needs review.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


### What it cost

This shows how many claims each entry got right, what it cost and how long it took for the whole queue.

In [70]:
show_misses(haikku_low, labels)

**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: Section 5 covers loss of personal belongings during the covered trip, and the sunglasses were lost on August 19 within the trip dates, so the loss is covered. However, the required police report is missing: the claimant asked the police post, which declined to take a report, and a guide letter was submitted instead, while section 5 requires a copy of the police report for theft. The theft story and the guide's letter also raise questions that need a colleague's review, so the claim is referred.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


In [71]:
scored.groupby(["model", "effort"], sort=False)[["right", "cost", "seconds"]].sum()

right      cost  seconds
model                        effort                          
anthropic/claude-haiku-5.5   low         4  0.003083     15.9
                             medium      3  0.003729     22.8
google/gemini-3.5-flash-lite low         3  0.008336      9.1

## Part 4: Using LangChain create_agent

Our approaches so far have used a single LLM call with Structured Outputs.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

In [ ]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

In [54]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

The agent uses Luna at `low` through LangChain's `ChatOpenAI`, pointed at OpenRouter. Its system prompt has the same instructions and policy wording as before, plus a line about the tools. Once the agent has looked up the policy it has the same information as before, so we can compare its score with Luna at `low`.

In [55]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=luna, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

First, trying out one claim. The model asks for the policy, the tool returns it, the model records its decision and then finishes with a short message.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()-

================================== Ai Message ==================================
Tool Calls:
  get_policy (call_33f1963e18f242ad91efcfc92e824ae6)
 Call ID: call_33f1963e18f242ad91efcfc92e824ae6
  Args:
    policy_id: POL-2007
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2007",
  "customer_id": "CUS-3007",
  "plan": "economy",
  "add_ons": [],
  "purchased_on": "2026-06-02",
  "sold_through": "Northwind Airlines checkout",
  "trip": {
    "destination": "Stockholm, Sweden",
    "departs_on": "2026-07-10",
    "returns_on": "2026-07-17",
    "trip_cost": 2200.0
  },
  "travelers": [
    "Greg Kowalski"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (call_yPhthPvT1PJ9yCKcdyazCL8g)
 Call ID: call_yPhthPvT1PJ9yCKcdyazCL8g
  Args:
    claim_id: CLM-1007
    decision: approve
    payout: 50
    rationale: The phone suffered acc

Now the whole queue. Each claim takes three model calls: one to ask for the policy, one to record the decision and one to finish. OpenRouter's cost for each call is in the message metadata. A claim with no recorded decision counts as wrong.

In [57]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": "openai/gpt-6-luna",
        "effort": "low",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

5 of 5 claims have a recorded decision


In [58]:
pd.DataFrame(luna_low + agent_results).groupby(["model", "effort", "technique"], sort=False)[["right", "cost", "seconds"]].sum()

right      cost  seconds
model             effort technique                          
openai/gpt-6-luna low    LLM            4  0.000773     12.7
                         Agent          3  0.001555     33.3

In [59]:
show_misses(agent_results, labels)

**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The reported loss occurred during the covered trip and could fall under Section 5, but theft must be reported to local police within 24 hours and a copy of the police report is required. The file contains only the guide’s letter, not a police report; under Section 11, the claim must wait for the required document or colleague review of the reported police refusal.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: The theft occurred during the covered trip, and the police report was made the same day, satisfying section 5’s theft-report requirement; the items were taken directly from the traveler, not left unattended. The eligible items total $1,103, within the Premium $1,500 benefit maximum and, with Electronics Plus, below the $1,200 electronics per-item limit; after the $50 deductible under sections 1 and 10, the payout is $1,053.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


## Putting it all together

Here is every experiment in one table. Cost is for the whole queue and speed is the average time for one claim. Jev isn't included because it only makes the decision.

In [61]:
# If you skipped Part 3, take sol out of this list
experiments = [luna_low, luna_medium, gemini, agent_results]

rows = []
for results in experiments:
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "reasoning": results[0]["effort"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "cost": f"${cost:.4f}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

,LLM,reasoning,technique,accuracy,cost,speed
0,openai/gpt-6-luna,low,LLM,4 / 5,$0.0008,2.5 seconds a claim
1,openai/gpt-6-luna,high,LLM,3 / 5,$0.0014,4.6 seconds a claim
2,google/gemini-3.5-flash-lite,low,LLM,3 / 5,$0.0083,1.8 seconds a claim
3,openai/gpt-6-luna,low,Agent,3 / 5,$0.0016,6.7 seconds a claim
